In [2]:
# CELL 1: Imports & Config
import re
import json
import time
import random
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import pandas as pd
import google.generativeai as genai

# ==== FILE PATHS ====
PATH_KEYWORDS = Path('keywords_fixed.json')
PATH_INPUT = Path('CRM_TDCTDA.xlsx')
PATH_OUTPUT = Path('CRM_TDCTDA_output.xlsx')
PATH_PROMPT = Path('prompt_CRM_v5.txt')

# ==== SOURCE COLUMNS ====
COL_R = 'Tình trạng hiện tại'
COL_S = 'Tình hình tiến độ công trình'
COL_T = 'Nội dung làm việc, yêu cầu KH & đánh giá'
COL_U = 'Kế hoạch lần tới'
COL_V = 'Đề xuất'

# ==== 15 OUTPUT COLUMNS (MAJOR, MINOR) ====
OUTPUT_SCHEMA = [
    ('Hoạt Động CRM', 'Tình hình hiện tại'),
    ('Hoạt Động CRM', 'Tiến độ'),
    ('Hoạt Động CRM', 'ngày lấy hàng'),
    ('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh'),
    ('Đối Thủ Cạnh Tranh', 'Đối tượng'),
    ('Đối Thủ Cạnh Tranh', 'Nội dung làm việc'),
    ('Đối Thủ Cạnh Tranh', 'Lợi thế'),
    ('AETT', 'Đối tượng'),
    ('AETT', 'Nội dung làm việc'),
    ('AETT', 'Nhận xét tiếp thị'),
    ('Khách Hàng', 'Ý kiến KH'),
    ('Khách Hàng', 'Nhận xét KH'),
    ('Kế Hoạch', 'Ngày'),
    ('Kế Hoạch', 'Kế hoạch lần tới'),
    ('Kế Hoạch', 'Đề xuất'),
]

def col_name(major: str, minor: str) -> str:
    return f'[{major}] {minor}'

OUTPUT_COLS = [col_name(m, f) for m, f in OUTPUT_SCHEMA]

# ==== API CONFIG ====
API_KEY = "YOUR_GEMINI_API_KEY_HERE"
MODEL_NAME = 'models/gemini-2.0-flash'

print('✓ Config loaded')
print(f'  - {len(OUTPUT_COLS)} output columns:')

✓ Config loaded
  - 15 output columns:


In [3]:
# CELL 2: Load Keywords & Build Pattern Map

with open(PATH_KEYWORDS, 'r', encoding='utf-8') as f:
    KW = json.load(f)

# ===== Lấy danh sách hãng TỰ ĐỘNG từ keywords_fixed.json =====
SPECIFIC_BRANDS: List[str] = []
GENERIC_BRAND_TERMS: List[str] = []

# keywords_fixed.json có 2 layout (hỗ trợ cả 2):
# - Layout A: Cạnh Tranh -> {"Các Hãng đối thủ cạnh tranh": [...] }
# - Layout B: Cạnh Tranh -> {"Hãng cạnh tranh": [...], "Các Hãng cụ thể": [...] }
competitor_section = KW.get('Đối Thủ Cạnh Tranh', {}).get('Cạnh Tranh', {})

brands_a = competitor_section.get('Các Hãng đối thủ cạnh tranh')
brands_b_specific = competitor_section.get('Các Hãng cụ thể')
brands_b_generic = competitor_section.get('Hãng cạnh tranh')

def _clean_list(xs):
    out = []
    if not isinstance(xs, list):
        return out
    for x in xs:
        if isinstance(x, str):
            s = x.strip()
            if s:
                out.append(s.lower())
    return out

if isinstance(brands_b_specific, list) or isinstance(brands_b_generic, list):
    # Layout B (ưu tiên vì tách rõ)
    SPECIFIC_BRANDS = _clean_list(brands_b_specific)
    GENERIC_BRAND_TERMS = _clean_list(brands_b_generic)
else:
    # Layout A (fallback)
    SPECIFIC_BRANDS = _clean_list(brands_a)
    GENERIC_BRAND_TERMS = []

# De-dup + đảm bảo hãng cụ thể không trùng từ chung chung (KHÔNG heuristic chuyển nhóm)
GENERIC_BRAND_TERMS = sorted(set(GENERIC_BRAND_TERMS))
SPECIFIC_BRANDS = sorted(set(SPECIFIC_BRANDS) - set(GENERIC_BRAND_TERMS))

print('✓ Extracted competitor terms from keywords_fixed.json:')
print(f'  - {len(SPECIFIC_BRANDS)} specific brands (sample: {", ".join(SPECIFIC_BRANDS[:8])})')
print(f'  - {len(GENERIC_BRAND_TERMS)} generic terms (sample: {", ".join(GENERIC_BRAND_TERMS[:8])})')

def compile_patterns(terms: List[str]) -> List[re.Pattern]:
    """Compile regex patterns - KHÔNG dùng unidecode để bắt chính xác"""
    pats: List[re.Pattern] = []
    for t in terms:
        t = t.strip()
        if not t:
            continue
        pats.append(re.compile(rf'(?<!\w){re.escape(t)}(?!\w)', re.IGNORECASE))
    return pats

# Build pattern map: {(major, minor, sub): [patterns]}
PATTERN_MAP: Dict[Tuple[str, str, str], List[re.Pattern]] = {}

for major, fields in KW.items():
    if not isinstance(fields, dict):
        continue
    for minor, subs in fields.items():
        # Bỏ qua phần hãng (đã xử lý ở trên)
        if major == 'Đối Thủ Cạnh Tranh' and minor == 'Cạnh Tranh':
            continue
        if isinstance(subs, dict):
            for sub, keywords in subs.items():
                if isinstance(keywords, list):
                    terms = [k.strip() for k in keywords if isinstance(k, str) and k.strip()]
                    if terms:
                        PATTERN_MAP[(major, minor, sub)] = compile_patterns(terms)

# Index để tìm nhanh theo (major, minor)
PATTERN_BY_FIELD: Dict[Tuple[str, str], List[Tuple[str, List[re.Pattern]]]] = {}
for (major, minor, sub), pats in PATTERN_MAP.items():
    PATTERN_BY_FIELD.setdefault((major, minor), []).append((sub, pats))

# Pre-compile brand patterns
SPECIFIC_BRAND_PATTERNS = [(b, compile_patterns([b])) for b in SPECIFIC_BRANDS]
GENERIC_BRAND_PATTERNS = compile_patterns(GENERIC_BRAND_TERMS)

# ===== Allowed values map for LLM (authoritative from keywords_fixed.json) =====
ALLOWED_BY_COL: Dict[str, List[str]] = {}
for major, minor in OUTPUT_SCHEMA:
    col = col_name(major, minor)
    if major == 'Đối Thủ Cạnh Tranh' and minor == 'Các Hãng đối thủ cạnh tranh':
        # multi-brand allowed; values must be in this list or exactly 'Hãng cạnh tranh'
        ALLOWED_BY_COL[col] = sorted(set(SPECIFIC_BRANDS)) + ['Hãng cạnh tranh']
        continue
    subs = KW.get(major, {}).get(minor, {})
    if isinstance(subs, dict):
        ALLOWED_BY_COL[col] = list(subs.keys())
    else:
        ALLOWED_BY_COL[col] = []

print(f'✓ Built ALLOWED_BY_COL for LLM: {len(ALLOWED_BY_COL)} columns')
print(f'  - competitor brands allowed: {len(ALLOWED_BY_COL.get(col_name("Đối Thủ Cạnh Tranh", "Các Hãng đối thủ cạnh tranh"), []))}')

print(f'✓ Loaded {len(PATTERN_MAP)} classification patterns')
print('✓ Built PATTERN_BY_FIELD index for faster matching')

✓ Extracted competitor terms from keywords_fixed.json:
  - 66 specific brands (sample: ac, anfaco, apolo, aqara, asia, cree, doepke, dow)
  - 11 generic terms (sample: bên khác, bên đấy, các dòng, các hãng, cân nhắc hãng, cạnh tranh, hãng cạnh tranh, hãng khác)
✓ Built ALLOWED_BY_COL for LLM: 15 columns
  - competitor brands allowed: 67
✓ Loaded 92 classification patterns
✓ Built PATTERN_BY_FIELD index for faster matching


In [4]:
# CELL 3: Load Data

df = pd.read_excel(PATH_INPUT)
print(f'✓ Loaded {len(df)} rows from {PATH_INPUT.name}')
print(f'  Columns: {[c for c in [COL_R, COL_S, COL_T, COL_U, COL_V] if c in df.columns]}')

✓ Loaded 21188 rows from CRM_TDCTDA.xlsx
  Columns: ['Tình trạng hiện tại', 'Tình hình tiến độ công trình', 'Nội dung làm việc, yêu cầu KH & đánh giá', 'Kế hoạch lần tới', 'Đề xuất']


In [5]:
# CELL 4: Helper Functions

def get_text(row, col: str) -> str:
    """Get text from column, return empty string if NaN"""
    val = row.get(col)
    if pd.isna(val):
        return ''
    s = str(val).strip()
    if s.lower() in ['nan', 'none']:
        return ''
    return s

def find_matches(text: str, major: str, minor: str) -> List[str]:
    """Find all matching subs for a given major/minor (FAST via index)"""
    if not text:
        return []
    items = PATTERN_BY_FIELD.get((major, minor), [])
    if not items:
        return []
    matches: List[str] = []
    for sub, pats in items:
        for p in pats:
            if p.search(text):
                matches.append(sub)
                break
    return matches

def find_brands(text: str) -> Tuple[List[str], bool]:
    """Find brand mentions. Returns: (specific_brands, has_generic)
    Notes: brands are returned EXACTLY as in keywords_fixed.json (no title-casing)
    """
    if not text:
        return [], False
    
    specific: List[str] = []
    for brand, pats in SPECIFIC_BRAND_PATTERNS:
        for p in pats:
            if p.search(text):
                specific.append(brand)
                break
    
    has_generic = any(p.search(text) for p in GENERIC_BRAND_PATTERNS)
    return specific, has_generic

def extract_date(text: str) -> Optional[str]:
    """Extract first date from text, normalize to dd/mm/yyyy. Default year: 2026."""
    if not text:
        return None
    
    # dd/mm/yyyy | dd-mm-yyyy | dd.mm.yyyy
    m = re.search(r'(\d{1,2})[\/.\-](\d{1,2})[\/.\-](\d{4})', text)
    if m:
        dd, mm, yyyy = m.groups()
        dd_i, mm_i, yyyy_i = int(dd), int(mm), int(yyyy)
        if 1 <= dd_i <= 31 and 1 <= mm_i <= 12:
            return f'{dd_i:02d}/{mm_i:02d}/{yyyy_i:04d}'
    
    # dd/mm/yy | dd-mm-yy | dd.mm.yy  -> assume 2000+yy
    m = re.search(r'(\d{1,2})[\/.\-](\d{1,2})[\/.\-](\d{2})(?!\d)', text)
    if m:
        dd, mm, yy = m.groups()
        dd_i, mm_i, yy_i = int(dd), int(mm), int(yy)
        if 1 <= dd_i <= 31 and 1 <= mm_i <= 12:
            yyyy_i = 2000 + yy_i
            return f'{dd_i:02d}/{mm_i:02d}/{yyyy_i:04d}'
    
    # dd/mm | dd-mm | dd.mm (assume 2026)
    m = re.search(r'(\d{1,2})[\/.\-](\d{1,2})(?![\/.\-]\d)', text)
    if m:
        dd, mm = m.groups()
        dd_i, mm_i = int(dd), int(mm)
        if 1 <= dd_i <= 31 and 1 <= mm_i <= 12:
            return f'{dd_i:02d}/{mm_i:02d}/2026'
    
    # tháng X[/YYYY]
    m = re.search(r'tháng\s*(\d{1,2})(?:\s*[\/.\-]\s*(\d{2,4}))?', text, re.IGNORECASE)
    if m:
        mm = int(m.group(1))
        yy_raw = m.group(2)
        if 1 <= mm <= 12:
            if yy_raw:
                yyyy_i = int(yy_raw)
                if yyyy_i < 100:
                    yyyy_i = 2000 + yyyy_i
            else:
                yyyy_i = 2026
            return f'01/{mm:02d}/{yyyy_i:04d}'
    
    # năm YYYY
    m = re.search(r'năm\s*(\d{4})', text, re.IGNORECASE)
    if m:
        yyyy = int(m.group(1))
        return f'01/01/{yyyy:04d}'
    
    return None

def has_eta_hint(text: str) -> bool:
    """Check if text has ETA-related keywords"""
    hints = ['giao hàng', 'cấp hàng', 'lấy hàng', 'eta', 'etd', 'xuất hàng']
    text_lower = text.lower()
    return any(h in text_lower for h in hints)

print('✓ Helper functions defined')

✓ Helper functions defined


In [6]:
# CELL 5: Main Classification Function

def classify_row(row) -> Dict[str, Any]:
    """
    Classify a single row.
    RULES:
    - Hoạt Động CRM, Đối Thủ, AETT, Khách Hàng: from R, S, T
    - Kế Hoạch: from U, V
    - Multi-match → 'mơ hồ'
    - Tình hình hiện tại: copy from R
    - IMPORTANT: Only assign Đối Thủ Cạnh Tranh sub-labels when brand column is detected
    """
    result = {}
    
    # Get source texts
    text_r = get_text(row, COL_R)
    text_s = get_text(row, COL_S)
    text_t = get_text(row, COL_T)
    text_u = get_text(row, COL_U)
    text_v = get_text(row, COL_V)
    
    text_rst = ' '.join([text_r, text_s, text_t]).strip()
    text_uv = ' '.join([text_u, text_v]).strip()
    
    # ===== 1. Hoạt Động CRM =====
    result[col_name('Hoạt Động CRM', 'Tình hình hiện tại')] = text_r if text_r else None
    
    matches = find_matches(text_rst, 'Hoạt Động CRM', 'Tiến độ')
    if len(matches) == 1:
        result[col_name('Hoạt Động CRM', 'Tiến độ')] = matches[0]
    elif len(matches) > 1:
        result[col_name('Hoạt Động CRM', 'Tiến độ')] = 'mơ hồ'
    
    if has_eta_hint(text_rst):
        date_val = extract_date(text_rst)
        if date_val:
            result[col_name('Hoạt Động CRM', 'ngày lấy hàng')] = date_val
    
    # ===== 2. Đối Thủ Cạnh Tranh =====
    specific_brands, has_generic = find_brands(text_rst)
    competitor_brand_val = None
    if specific_brands:
        competitor_brand_val = '; '.join(specific_brands)
    elif has_generic:
        competitor_brand_val = 'Hãng cạnh tranh'
    
    # Chỉ khi bắt được hãng thì mới gán nhãn các cột con của Đối Thủ Cạnh Tranh
    if competitor_brand_val:
        result[col_name('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh')] = competitor_brand_val
        
        for minor in ['Đối tượng', 'Nội dung làm việc', 'Lợi thế']:
            matches = find_matches(text_rst, 'Đối Thủ Cạnh Tranh', minor)
            if len(matches) == 1:
                result[col_name('Đối Thủ Cạnh Tranh', minor)] = matches[0]
            elif len(matches) > 1:
                result[col_name('Đối Thủ Cạnh Tranh', minor)] = 'mơ hồ'
    
    # ===== 3. AETT =====
    for minor in ['Đối tượng', 'Nội dung làm việc', 'Nhận xét tiếp thị']:
        matches = find_matches(text_rst, 'AETT', minor)
        if len(matches) == 1:
            result[col_name('AETT', minor)] = matches[0]
        elif len(matches) > 1:
            result[col_name('AETT', minor)] = 'mơ hồ'
    
    # ===== 4. Khách Hàng =====
    for minor in ['Ý kiến KH', 'Nhận xét KH']:
        matches = find_matches(text_rst, 'Khách Hàng', minor)
        if len(matches) == 1:
            result[col_name('Khách Hàng', minor)] = matches[0]
        elif len(matches) > 1:
            result[col_name('Khách Hàng', minor)] = 'mơ hồ'
    
    # ===== 5. Kế Hoạch (from U, V) =====
    date_val = extract_date(text_uv)
    if date_val:
        result[col_name('Kế Hoạch', 'Ngày')] = date_val
    
    matches = find_matches(text_u, 'Kế Hoạch', 'Kế hoạch lần tới')
    if len(matches) == 1:
        result[col_name('Kế Hoạch', 'Kế hoạch lần tới')] = matches[0]
    elif len(matches) > 1:
        result[col_name('Kế Hoạch', 'Kế hoạch lần tới')] = 'mơ hồ'
    
    matches = find_matches(text_v, 'Kế Hoạch', 'Đề xuất')
    if len(matches) == 1:
        result[col_name('Kế Hoạch', 'Đề xuất')] = matches[0]
    elif len(matches) > 1:
        result[col_name('Kế Hoạch', 'Đề xuất')] = 'mơ hồ'
    
    return result

print('✓ classify_row() defined')

✓ classify_row() defined


In [7]:
# CELL 6: Run Keyword Classification

print('Starting keyword classification...')
all_results = []
fuzzy_count = 0

for idx, row in df.iterrows():
    if idx % 2000 == 0:
        print(f'  {idx}/{len(df)}')
    
    cls = classify_row(row)
    cls['_row_idx'] = idx
    all_results.append(cls)
    
    # Count fuzzy
    for v in cls.values():
        if v == 'mơ hồ':
            fuzzy_count += 1
            break

print(f'✓ Classified {len(all_results)} rows')
print(f'  - {fuzzy_count} rows have at least one "mơ hồ" cell')

# Convert to DataFrame
df_cls = pd.DataFrame(all_results)
df_cls.head()

Starting keyword classification...
  0/21188
  2000/21188
  4000/21188
  6000/21188
  8000/21188
  10000/21188
  12000/21188
  14000/21188
  16000/21188
  18000/21188
  20000/21188
✓ Classified 21188 rows
  - 9543 rows have at least one "mơ hồ" cell


,[Hoạt Động CRM] Tình hình hiện tại,_row_idx,[AETT] Nội dung làm việc,[Kế Hoạch] Kế hoạch lần tới,[AETT] Đối tượng,[Khách Hàng] Ý kiến KH,[Khách Hàng] Nhận xét KH,[Hoạt Động CRM] Tiến độ,[Đối Thủ Cạnh Tranh] Các Hãng đối thủ cạnh tranh,[Đối Thủ Cạnh Tranh] Đối tượng,[Đối Thủ Cạnh Tranh] Nội dung làm việc,[Đối Thủ Cạnh Tranh] Lợi thế,[Kế Hoạch] Ngày,[Hoạt Động CRM] ngày lấy hàng,[Kế Hoạch] Đề xuất,[AETT] Nhận xét tiếp thị
0,None,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Thi công điện nước,1,mơ hồ,Thương thảo hợp đồng,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Thi công điện nước,2,mơ hồ,Chốt giá,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Thi công điện nước,3,mơ hồ,Chốt giá,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# CELL 7: Prepare LLM Input

def prepare_llm_input(row_idx: int, row_data: Dict[str, Any], orig_row) -> Optional[Dict]:
    """Prepare input for LLM - ONLY when there is at least one 'mơ hồ' cell."""
    fuzzy_cols: List[str] = []
    locked_labels: Dict[str, Any] = {}
    
    for col in OUTPUT_COLS:
        if col == col_name('Hoạt Động CRM', 'Tình hình hiện tại'):
            continue  # Always copy from R
        
        val = row_data.get(col)
        if val == 'mơ hồ':
            fuzzy_cols.append(col)
        elif val not in (None, '', 0, '0'):
            locked_labels[col] = val
    
    if not fuzzy_cols:
        return None
    
    text_r = get_text(orig_row, COL_R)
    text_s = get_text(orig_row, COL_S)
    text_t = get_text(orig_row, COL_T)
    text_u = get_text(orig_row, COL_U)
    text_v = get_text(orig_row, COL_V)
    
    if not any([text_r, text_s, text_t, text_u, text_v]):
        return None
    
    allowed = {c: ALLOWED_BY_COL.get(c, []) for c in fuzzy_cols}
    out = {
        'idx': row_idx,
        'R': text_r,
        'S': text_s,
        'T': text_t,
        'U': text_u,
        'V': text_v,
        'missing_cols': fuzzy_cols,
        'allowed': allowed,
    }
    if locked_labels:
        out['locked_labels'] = locked_labels
    return out

llm_items = []
for idx, cls in enumerate(all_results):
    orig_row = df.iloc[idx]
    item = prepare_llm_input(idx, cls, orig_row)
    if item:
        llm_items.append(item)

print(f'✓ {len(llm_items)} rows need LLM processing')

✓ 9543 rows need LLM processing


In [9]:
# CELL 8: LLM Setup

SYSTEM_PROMPT = PATH_PROMPT.read_text(encoding='utf-8')

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(MODEL_NAME)

# Batching notes: Gemini has large context, but output is limited (max_output_tokens).
# Batch 50 reduces calls, but if JSON truncates we auto-split in CELL 9.
MIN_INTERVAL = 2.5
BATCH_SIZE = 50
last_call_time = 0

def throttle():
    global last_call_time
    elapsed = time.time() - last_call_time
    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed + random.random() * 0.5)
    last_call_time = time.time()

def extract_json_array(text: str) -> List[Dict]:
    """Extract JSON array from LLM response"""
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```json?\s*', '', text)
        text = re.sub(r'```$', '', text)
    text = text.strip()
    
    start = text.find('[')
    if start < 0:
        return []
    
    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '[':
                depth += 1
            elif ch == ']':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i+1])
                    except:
                        return []
    return []

def _pack_item(it: Dict[str, Any]) -> Dict[str, Any]:
    """Reduce token usage: drop empty fields and keep only needed locked labels."""
    out = {
        'idx': it.get('idx'),
        'missing_cols': it.get('missing_cols', []),
        'allowed': it.get('allowed', {}),
    }
    for k in ['R', 'S', 'T', 'U', 'V']:
        v = it.get(k, '')
        if isinstance(v, str) and v.strip():
            out[k] = v
    locked = it.get('locked_labels')
    if isinstance(locked, dict) and locked:
        out['locked_labels'] = locked
    return out

def call_llm_batch(batch: List[Dict]) -> List[Dict]:
    """Call LLM for a batch. Returns list of {idx, fills, evidence}."""
    packed = [_pack_item(x) for x in batch]
    payload = json.dumps(packed, ensure_ascii=False)
    user_input = f'Process these {len(packed)} items:\n{payload}'
    
    for attempt in range(3):
        try:
            throttle()
            resp = model.generate_content(
                SYSTEM_PROMPT + '\n\n' + user_input,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.0,
                    max_output_tokens=4096,
                    response_mime_type='application/json',
                ),
            )
            raw = getattr(resp, 'text', '') or ''
            try:
                return json.loads(raw) if raw.strip().startswith('[') else extract_json_array(raw)
            except:
                return extract_json_array(raw)
        except Exception as e:
            msg = str(e).lower()
            if '429' in msg or 'rate' in msg:
                wait = 15 * (attempt + 1)
                print(f'  Rate limit, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'  Error: {e}')
                time.sleep(5)
    return []

print(f'✓ LLM ready: {MODEL_NAME}')
print(f'  - Batch size: {BATCH_SIZE}')
print(f'  - Items to process: {len(llm_items)}')

✓ LLM ready: models/gemini-2.0-flash
  - Batch size: 50
  - Items to process: 9543


In [10]:
# CELL 9: Run LLM Processing (batch=50 with auto-split fallback)

llm_results = {}

def _ingest_results(results: List[Dict]):
    if not isinstance(results, list):
        return
    for r in results:
        if isinstance(r, dict) and 'idx' in r:
            llm_results[int(r['idx'])] = r

def process_batch(batch: List[Dict], depth: int = 0):
    """Run a batch; if response is empty/partial, split to keep accuracy."""
    results = call_llm_batch(batch)
    if not batch:
        return
    want = {int(x.get('idx')) for x in batch if x.get('idx') is not None}
    got = {int(x.get('idx')) for x in results if isinstance(x, dict) and x.get('idx') is not None}
    # Success if we got all idx back
    if want and want.issubset(got):
        _ingest_results(results)
        return
    # If failed and can split, recurse
    if len(batch) > 1 and depth < 6:
        mid = len(batch) // 2
        process_batch(batch[:mid], depth + 1)
        process_batch(batch[mid:], depth + 1)
        return
    # Last resort: ingest whatever we got
    _ingest_results(results)

print('Starting LLM processing...')
total_batches = (len(llm_items) + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(0, len(llm_items), BATCH_SIZE):
    batch = llm_items[i:i + BATCH_SIZE]
    print(f'  Batch {i//BATCH_SIZE + 1}/{total_batches} ({len(batch)} items)')
    process_batch(batch)

print(f'\n✓ LLM processing complete')
print(f'  - Processed: {len(llm_results)} rows')

Starting LLM processing...
  Batch 1/191 (50 items)
  Batch 2/191 (50 items)
  Batch 3/191 (50 items)
  Batch 4/191 (50 items)
  Batch 5/191 (50 items)
  Batch 6/191 (50 items)
  Batch 7/191 (50 items)
  Batch 8/191 (50 items)
  Batch 9/191 (50 items)
  Batch 10/191 (50 items)
  Batch 11/191 (50 items)
  Batch 12/191 (50 items)
  Batch 13/191 (50 items)
  Batch 14/191 (50 items)
  Batch 15/191 (50 items)
  Batch 16/191 (50 items)
  Batch 17/191 (50 items)
  Batch 18/191 (50 items)
  Batch 19/191 (50 items)
  Batch 20/191 (50 items)
  Batch 21/191 (50 items)
  Batch 22/191 (50 items)
  Batch 23/191 (50 items)
  Batch 24/191 (50 items)
  Batch 25/191 (50 items)
  Batch 26/191 (50 items)
  Batch 27/191 (50 items)
  Batch 28/191 (50 items)
  Batch 29/191 (50 items)
  Batch 30/191 (50 items)
  Batch 31/191 (50 items)
  Batch 32/191 (50 items)
  Batch 33/191 (50 items)
  Batch 34/191 (50 items)
  Batch 35/191 (50 items)
  Batch 36/191 (50 items)
  Batch 37/191 (50 items)
  Rate limit, waitin

In [11]:
# CELL 10: Merge Results, Enforce Rules, Fill 0

# Allow skipping CELL 9 (LLM) by defaulting to empty results
llm_results = llm_results if 'llm_results' in globals() and isinstance(llm_results, dict) else {}

AETT_WORK_COL = col_name('AETT', 'Nội dung làm việc')
COMP_BRAND_COL = col_name('Đối Thủ Cạnh Tranh', 'Các Hãng đối thủ cạnh tranh')

def _is_blank_value(v: Any) -> bool:
    return v is None or v == '' or v == 0 or v == '0'

def _normalize_llm_val(v: Any) -> Optional[str]:
    if v is None:
        return None
    s = str(v).strip()
    if not s or s.lower() in {'null', 'none', 'nan'}:
        return None
    return s

def _is_valid_fill(col: str, val: Any) -> bool:
    """Validate LLM output against allowed taxonomy (per-column)."""
    s = _normalize_llm_val(val)
    if s is None:
        return False
    allowed = ALLOWED_BY_COL.get(col, [])
    if col == COMP_BRAND_COL:
        if s == 'Hãng cạnh tranh':
            return True
        parts = [p.strip() for p in s.split(';') if p.strip()]
        if not parts:
            return False
        allowed_set = set(allowed)
        return all(p in allowed_set for p in parts)
    return s in set(allowed)

for idx, llm_data in llm_results.items():
    fills = llm_data.get('fills', {}) or {}
    evidence = llm_data.get('evidence', {})
    
    if idx < len(all_results):
        # Apply fills (only overwrite if current is None/'mơ hồ')
        applied = {}
        for col, val in fills.items():
            if col in OUTPUT_COLS:
                current = all_results[idx].get(col)
                if current is None or current == 'mơ hồ':
                    if _is_valid_fill(col, val):
                        all_results[idx][col] = _normalize_llm_val(val)
                        applied[col] = _normalize_llm_val(val)
        
        # Build evidence string that names which tag/value it supports
        # evidence can be dict (preferred) or string (legacy)
        if isinstance(evidence, dict):
            parts = []
            for col, val in applied.items():
                ev = evidence.get(col, '')
                parts.append(f'{col}={val} :: {ev}'.strip())
            all_results[idx]['_evidence'] = ' | '.join([p for p in parts if p])
        else:
            fills_summary = '; '.join([f'{c}={v}' for c, v in applied.items()])
            base = str(evidence or '').strip()
            all_results[idx]['_evidence'] = (fills_summary + (' | ' + base if base else '')).strip(' |')

# Enforce rule: if raw "Nội dung làm việc" (COL_T) has text, then [AETT] Nội dung làm việc must be label OR 'mơ hồ' (never 0)
for i, cls in enumerate(all_results):
    text_t = get_text(df.iloc[i], COL_T)
    if text_t:
        v = cls.get(AETT_WORK_COL)
        if _is_blank_value(v):
            cls[AETT_WORK_COL] = 'mơ hồ'

# Fill remaining blanks/mơ hồ with 0, EXCEPT enforced AETT work column when COL_T has content
for i, cls in enumerate(all_results):
    text_t = get_text(df.iloc[i], COL_T)
    for col in OUTPUT_COLS:
        val = cls.get(col)
        if col == AETT_WORK_COL and text_t:
            # keep as-is; if still blank, force 'mơ hồ'
            if _is_blank_value(val):
                cls[col] = 'mơ hồ'
            continue
        if val is None or val == 'mơ hồ':
            cls[col] = 0

print('✓ Merged LLM results (validated), enforced AETT rule, and filled 0 for other empty cells')

✓ Merged LLM results (validated), enforced AETT rule, and filled 0 for other empty cells


In [12]:
# CELL 11: Export to Excel (formatted)

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

df_output = df.copy()

for col in OUTPUT_COLS:
    df_output[col] = [cls.get(col, 0) for cls in all_results]

df_output['Bằng chứng'] = [cls.get('_evidence', '') for cls in all_results]

# Write raw dataframe first
df_output.to_excel(PATH_OUTPUT, index=False)

# Apply basic formatting to make columns readable
wb = load_workbook(PATH_OUTPUT)
ws = wb.active

# Build 2-row grouped header
headers = [cell.value for cell in ws[1]]
def _split_header(h: str) -> Tuple[str, str]:
    if isinstance(h, str) and h.startswith('[') and '] ' in h:
        major = h[1:h.find(']')].strip()
        minor = h[h.find(']')+1:].strip()
        return major, minor
    return 'RAW', str(h)

groups = []
minors = []
for h in headers:
    g, m = _split_header(h)
    groups.append(g)
    minors.append(m)

# Insert a new top row for group names
ws.insert_rows(1)

# Write group row (row 1) and minor row (row 2)
for j, (g, m) in enumerate(zip(groups, minors), start=1):
    ws.cell(row=1, column=j, value=g)
    ws.cell(row=2, column=j, value=m)

# Merge contiguous group cells in row 1
start = 1
for j in range(2, len(groups) + 2):
    if j == len(groups) + 1 or groups[j-1] != groups[start-1]:
        if j - 1 > start:
            ws.merge_cells(start_row=1, start_column=start, end_row=1, end_column=j-1)
        start = j

# Style header rows
header_font = Font(bold=True)
header_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
for row in (1, 2):
    ws.row_dimensions[row].height = 28
    for j in range(1, len(headers) + 1):
        c = ws.cell(row=row, column=j)
        c.font = header_font
        c.alignment = header_align

# Freeze panes below headers
ws.freeze_panes = ws['A3']

# Auto-filter on row 2
ws.auto_filter.ref = ws.dimensions

# Column widths (simple heuristics)
for j, h in enumerate(headers, start=1):
    col_letter = get_column_letter(j)
    base = minors[j-1] if minors else str(h)
    width = max(14, min(60, int(len(str(base)) * 1.2) + 8))
    if base in [COL_R, COL_S, COL_T, COL_U, COL_V]:
        width = 45
    if str(h) == 'Bằng chứng':
        width = 70
    ws.column_dimensions[col_letter].width = width

# Wrap text for content-heavy columns
wrap_cols = {COL_R, COL_S, COL_T, COL_U, COL_V, 'Bằng chứng'}
for j, h in enumerate(headers, start=1):
    base = minors[j-1] if minors else str(h)
    if base in wrap_cols or str(h) in wrap_cols:
        for cell in ws[get_column_letter(j)]:
            cell.alignment = Alignment(vertical='top', wrap_text=True)

wb.save(PATH_OUTPUT)

print(f'✓ Exported formatted Excel to {PATH_OUTPUT}')
print(f'  - Shape: {df_output.shape}')

✓ Exported formatted Excel to CRM_TDCTDA_output.xlsx
  - Shape: (21188, 38)


In [13]:
# CELL 12: Summary

print('=== SUMMARY ===')
print(f'Total rows: {len(df_output)}')
print()

for col in OUTPUT_COLS:
    non_zero = (df_output[col] != 0).sum()
    pct = non_zero / len(df_output) * 100
    print(f'{col}: {non_zero} ({pct:.1f}%)')

print()
print(f'Evidence filled: {(df_output["Bằng chứng"] != "").sum()}')

=== SUMMARY ===
Total rows: 21188

[Hoạt Động CRM] Tình hình hiện tại: 9333 (44.0%)
[Hoạt Động CRM] Tiến độ: 553 (2.6%)
[Hoạt Động CRM] ngày lấy hàng: 444 (2.1%)
[Đối Thủ Cạnh Tranh] Các Hãng đối thủ cạnh tranh: 623 (2.9%)
[Đối Thủ Cạnh Tranh] Đối tượng: 461 (2.2%)
[Đối Thủ Cạnh Tranh] Nội dung làm việc: 517 (2.4%)
[Đối Thủ Cạnh Tranh] Lợi thế: 434 (2.0%)
[AETT] Đối tượng: 4292 (20.3%)
[AETT] Nội dung làm việc: 14918 (70.4%)
[AETT] Nhận xét tiếp thị: 1049 (5.0%)
[Khách Hàng] Ý kiến KH: 190 (0.9%)
[Khách Hàng] Nhận xét KH: 150 (0.7%)
[Kế Hoạch] Ngày: 565 (2.7%)
[Kế Hoạch] Kế hoạch lần tới: 4604 (21.7%)
[Kế Hoạch] Đề xuất: 175 (0.8%)

Evidence filled: 9359
